
# **ML Modelling**

In [1]:
# LOAD DATA
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import glob, os, joblib

folders = glob.glob("/content/drive/MyDrive/data_after_corr_*")

if len(folders) == 0:
    raise ValueError("No saved dataset found. Check your folder name.")

latest_folder = max(folders, key=os.path.getmtime)
print("Loading data from:", latest_folder)

# FIX: correct filenames
X_train = joblib.load(os.path.join(latest_folder, "X_train_final.pkl"))
X_test = joblib.load(os.path.join(latest_folder, "X_test_final.pkl"))
y_train = joblib.load(os.path.join(latest_folder, "y_train.pkl"))
y_test = joblib.load(os.path.join(latest_folder, "y_test.pkl"))


# LOAD FEATURE SELECTION RESULTS
fs_folders = glob.glob("/content/drive/MyDrive/FS_results_*")

if len(fs_folders) == 0:
    raise ValueError("No FS results found.")

latest_fs_folder = max(fs_folders, key=os.path.getmtime)
print("Loading FS from:", latest_fs_folder)

fs_methods = {}

for file in os.listdir(latest_fs_folder):
    if file.endswith(".pkl"):
        name = file.replace(".pkl", "")
        fs_methods[name] = joblib.load(os.path.join(latest_fs_folder, file))

# ADD full dataset (no FS)
fs_methods["ALL_FEATURES"] = X_train.columns.tolist()


# SMOTE (TRAIN ONLY)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42)

# Import
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

# EVALUATION METRICS FUNCTION
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_test, y_pred):
    return {
        "Accuracy": accuracy_score(y_test, y_pred) * 100,
        "Precision (W)": precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "Recall (W)": recall_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "F1 (W)": f1_score(y_test, y_pred, average='weighted', zero_division=0) * 100,
        "Precision (Macro)": precision_score(y_test, y_pred, average='macro', zero_division=0) * 100,
        "Recall (Macro)": recall_score(y_test, y_pred, average='macro', zero_division=0) * 100,
        "F1 (Macro)": f1_score(y_test, y_pred, average='macro', zero_division=0) * 100,
    }

def print_results(results):
    for k, v in results.items():
        print(f"{k}: {v:.2f}%")

Mounted at /content/drive
Loading data from: /content/drive/MyDrive/data_after_corr_20260507_1258
Loading FS from: /content/drive/MyDrive/FS_results_20260507_1258


In [2]:
# Print Loaded Feature Sets

print("\n================ FEATURE SETS LOADED ================")

for method, feats in fs_methods.items():
    print(f"\n{method}:")
    print(f"Number of features: {len(feats)}")
    print("Features:", feats)



================ FEATURE SETS LOADED ================

ANOVA:
Number of features: 15
Features: ['RISK  - Risk Type', 'Target HR (bpm)', 'Test Today - METS', 'Functional Activity', 'Risk Factor - ECHO - EF', 'Age', 'Smoking', 'Family History', 'Risk Factor - Family hx', 'Gait', 'Exercise Habit - Duration', 'Occupation', 'Balance in Sitting and Standing', 'Test Today - Termination Cause', 'Exercise Habit - Frequency']

Mutual_Info:
Number of features: 15
Features: ['RISK  - Risk Type', 'Target HR (bpm)', 'Risk Factor - ECHO - EF', 'Posture', 'Gender', 'ECG Resting', 'Family History', 'Year', 'Cooling Down', 'Test Today - METS', 'Risk Factor - Stress', 'Risk Factor - Exercise', 'Smoking', 'Risk Factor - Family hx', 'Balance in Sitting and Standing']

RFE_LR:
Number of features: 15
Features: ['RISK  - Risk Type', 'Exercise Habit - Frequency', 'Exercise Habit - Mode', 'Walking', 'Diagnosis', 'Gait', 'Risk Factor - DM', 'Total_Muscle_Power', 'Smoking', 'Functional Activity', 'Posture', 'ROM

# **Conventional Models**

*   Decision Tree
*   Logistic Regression (Classifier)
*   k-Nearest Neighbors (k-NN)


In [ ]:
# CONVENTIONAL MODELS

import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# CV (STANDARDIZED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# MODELS
models = {

    # Decision Tree (NO scaling, NO ROS)
    "Decision Tree": (
        DecisionTreeClassifier(
            class_weight='balanced',
            random_state=42
        ),
        {
            "max_depth": list(range(2, 30))
        }
    ),

    # Logistic Regression
    "Logistic Regression": (
        ImbPipeline([
            ("ros", RandomOverSampler(random_state=42)),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=5000))
        ]),
        {
            "model__C": np.logspace(-3, 2, 10)
        }
    ),

    # k-NN
    "k-NN": (
        ImbPipeline([
            ("ros", RandomOverSampler(random_state=42)),
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())
        ]),
        {
            "model__n_neighbors": list(range(3, 15))
        }
    ),
}

# LOOP
for fs_name, features in fs_methods.items():

    print(f"\n\n========== CONVENTIONAL | FEATURE SET: {fs_name} ==========")

    X_train_sel = X_train[features]
    X_test_sel = X_test[features]

    for name, (model, params) in models.items():

        print(f"\n{name}")

        # Avoid n_iter warning
        total_space = np.prod([len(v) for v in params.values()])
        n_iter = min(10, total_space)

        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=params,
            n_iter=n_iter,
            cv=cv,
            scoring='accuracy',
            n_jobs=-1,
            random_state=42
        )

        # TRAIN (NO LEAKAGE — ROS inside pipeline)
        search.fit(X_train_sel, y_train)

        # TEST
        y_pred = search.predict(X_test_sel)

        print("Best Params:", search.best_params_)
        print_results(evaluate(y_test, y_pred))



========== CONVENTIONAL | FEATURE SET: ANOVA ==========

Decision Tree
Best Params: {'max_depth': 11}
Accuracy: 87.88%
Precision (W): 83.98%
Recall (W): 87.88%
F1 (W): 85.82%
Precision (Macro): 58.40%
Recall (Macro): 61.82%
F1 (Macro): 60.01%

Logistic Regression
Best Params: {'model__C': np.float64(0.5994842503189409)}
Accuracy: 77.27%
Precision (W): 90.98%
Recall (W): 77.27%
F1 (W): 81.70%
Precision (Macro): 67.58%
Recall (Macro): 76.15%
F1 (Macro): 65.26%

k-NN
Best Params: {'model__n_neighbors': 3}
Accuracy: 83.33%
Precision (W): 79.84%
Recall (W): 83.33%
F1 (W): 81.40%
Precision (Macro): 55.27%
Recall (Macro): 58.74%
F1 (Macro): 56.84%

SVM-RBF
Best Params: {'model__gamma': 'scale', 'model__C': 10}
Accuracy: 89.39%
Precision (W): 86.67%
Recall (W): 89.39%
F1 (W): 88.00%
Precision (Macro): 60.67%
Recall (Macro): 62.25%
F1 (Macro): 61.44%


========== CONVENTIONAL | FEATURE SET: Mutual_Info ==========

Decision Tree
Best Params: {'max_depth': 27}
Accuracy: 84.85%
Precision (W): 84

# **Ensemble models**

*   Random Forest
*   XGBoost
*   LightGBM


In [ ]:
# ENSEMBLE MODELS

import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# INSTALL MISSING PACKAGES (FIX)
import sys
import subprocess

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    from lightgbm import LGBMClassifier
except ModuleNotFoundError:
    install("lightgbm")
    from lightgbm import LGBMClassifier

import xgboost as xgb

# CV (STANDARDIZED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# MODELS
ensembles = {

    # Random Forest
    "Random Forest": (
        RandomForestClassifier(
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ),
        {
            "n_estimators": [100, 200],
            "max_depth": [None, 10, 20],
        }
    ),

    # LightGBM
    "LightGBM": (
        LGBMClassifier(
            class_weight='balanced',
            random_state=42,
            verbosity=-1
        ),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
        }
    ),

    # XGBoost
    "XGBoost": (
        xgb.XGBClassifier(
            eval_metric='mlogloss',
            random_state=42,
            use_label_encoder=False
        ),
        {
            "n_estimators": [100, 200],
            "max_depth": [3, 5],
            "learning_rate": [0.05, 0.1],
        }
    ),
}

# LOOP
for fs_name, features in fs_methods.items():

    print(f"\n\n========== ENSEMBLE | FEATURE SET: {fs_name} ==========")

    X_train_sel = X_train[features]
    X_test_sel = X_test[features]

    for name, (model, params) in ensembles.items():

        print(f"\n{name}")

        # Prevent n_iter warning (already correct)
        total_space = int(np.prod([len(v) for v in params.values()]))
        n_iter = min(10, total_space)

        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=params,
            n_iter=n_iter,
            cv=cv,
            scoring='accuracy',
            n_jobs=-1 if name != "CatBoost" else None,
            random_state=42
        )

        # TRAIN
        search.fit(X_train_sel, y_train)

        # TEST
        y_pred = search.predict(X_test_sel)

        # RESULTS
        print("Best Params:", search.best_params_)
        print_results(evaluate(y_test, y_pred))



========== ENSEMBLE | FEATURE SET: RFE_SVM ==========

Random Forest
Best Params: {'n_estimators': 200, 'max_depth': None}
Accuracy: 92.42%
Precision (W): 88.22%
Recall (W): 92.42%
F1 (W): 90.26%
Precision (Macro): 61.63%
Recall (Macro): 64.91%
F1 (Macro): 63.22%

LightGBM
Best Params: {'n_estimators': 100, 'learning_rate': 0.1}
Accuracy: 89.39%
Precision (W): 86.64%
Recall (W): 89.39%
F1 (W): 87.99%
Precision (Macro): 60.68%
Recall (Macro): 62.70%
F1 (Macro): 61.68%

XGBoost


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:36:36] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Params: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05}
Accuracy: 90.91%
Precision (W): 86.78%
Recall (W): 90.91%
F1 (W): 88.80%
Precision (Macro): 60.77%
Recall (Macro): 63.58%
F1 (Macro): 62.14%

CatBoost
Best Params: {'learning_rate': 0.1, 'iterations': 200, 'depth': 6}
Accuracy: 87.88%
Precision (W): 86.48%
Recall (W): 87.88%
F1 (W): 87.17%
Precision (Macro): 60.59%
Recall (Macro): 61.82%
F1 (Macro): 61.20%


========== ENSEMBLE | FEATURE SET: Mutual_Info ==========

Random Forest
Best Params: {'n_estimators': 100, 'max_depth': None}
Accuracy: 90.91%
Precision (W): 86.78%
Recall (W): 90.91%
F1 (W): 88.80%
Precision (Macro): 60.77%
Recall (Macro): 63.58%
F1 (Macro): 62.14%

LightGBM
Best Params: {'n_estimators': 200, 'learning_rate': 0.05}
Accuracy: 90.91%
Precision (W): 86.78%
Recall (W): 90.91%
F1 (W): 88.80%
Precision (Macro): 60.77%
Recall (Macro): 63.58%
F1 (Macro): 62.14%

XGBoost


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:37:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Params: {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05}
Accuracy: 87.88%
Precision (W): 85.19%
Recall (W): 87.88%
F1 (W): 86.48%
Precision (Macro): 59.45%
Recall (Macro): 61.82%
F1 (Macro): 60.59%

CatBoost
Best Params: {'learning_rate': 0.1, 'iterations': 400, 'depth': 6}
Accuracy: 90.91%
Precision (W): 86.78%
Recall (W): 90.91%
F1 (W): 88.80%
Precision (Macro): 60.77%
Recall (Macro): 63.58%
F1 (Macro): 62.14%


========== ENSEMBLE | FEATURE SET: RFE_LR ==========

Random Forest
Best Params: {'n_estimators': 200, 'max_depth': None}
Accuracy: 90.91%
Precision (W): 86.78%
Recall (W): 90.91%
F1 (W): 88.80%
Precision (Macro): 60.77%
Recall (Macro): 63.58%
F1 (Macro): 62.14%

LightGBM
Best Params: {'n_estimators': 200, 'learning_rate': 0.1}
Accuracy: 89.39%
Precision (W): 86.64%
Recall (W): 89.39%
F1 (W): 87.99%
Precision (Macro): 60.68%
Recall (Macro): 62.70%
F1 (Macro): 61.68%

XGBoost


...

Best Params: {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05}
Accuracy: 92.42%
Precision (W): 92.66%
Recall (W): 92.42%
F1 (W): 91.76%
Precision (Macro): 94.87%
Recall (Macro): 74.69%
F1 (Macro): 79.21%

CatBoost
Best Params: {'learning_rate': 0.1, 'iterations': 400, 'depth': 4}
Accuracy: 92.42%
Precision (W): 92.66%
Recall (W): 92.42%
F1 (W): 91.76%
Precision (Macro): 94.87%
Recall (Macro): 74.69%
F1 (Macro): 79.21%


========== ENSEMBLE | FEATURE SET: RF ==========

Random Forest
Best Params: {'n_estimators': 100, 'max_depth': None}
Accuracy: 90.91%
Precision (W): 86.78%
Recall (W): 90.91%
F1 (W): 88.80%
Precision (Macro): 60.77%
Recall (Macro): 63.58%
F1 (Macro): 62.14%

LightGBM
Best Params: {'n_estimators': 200, 'learning_rate': 0.05}
Accuracy: 92.42%
Precision (W): 92.66%
Recall (W): 92.42%
F1 (W): 91.76%
Precision (Macro): 94.87%
Recall (Macro): 74.69%
F1 (Macro): 79.21%

XGBoost


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:39:06] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Params: {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05}
Accuracy: 90.91%
Precision (W): 86.78%
Recall (W): 90.91%
F1 (W): 88.80%
Precision (Macro): 60.77%
Recall (Macro): 63.58%
F1 (Macro): 62.14%

CatBoost
Best Params: {'learning_rate': 0.05, 'iterations': 200, 'depth': 6}
Accuracy: 90.91%
Precision (W): 86.78%
Recall (W): 90.91%
F1 (W): 88.80%
Precision (Macro): 60.77%
Recall (Macro): 63.58%
F1 (Macro): 62.14%


========== ENSEMBLE | FEATURE SET: ALL_FEATURES ==========

Random Forest
Best Params: {'n_estimators': 100, 'max_depth': 10}
Accuracy: 90.91%
Precision (W): 86.78%
Recall (W): 90.91%
F1 (W): 88.80%
Precision (Macro): 60.77%
Recall (Macro): 63.58%
F1 (Macro): 62.14%

LightGBM
Best Params: {'n_estimators': 200, 'learning_rate': 0.05}
Accuracy: 92.42%
Precision (W): 92.66%
Recall (W): 92.42%
F1 (W): 91.76%
Precision (Macro): 94.87%
Recall (Macro): 74.69%
F1 (Macro): 79.21%

XGBoost


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:39:44] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Params: {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05}
Accuracy: 90.91%
Precision (W): 86.78%
Recall (W): 90.91%
F1 (W): 88.80%
Precision (Macro): 60.77%
Recall (Macro): 63.58%
F1 (Macro): 62.14%

CatBoost
Best Params: {'learning_rate': 0.05, 'iterations': 200, 'depth': 6}
Accuracy: 90.91%
Precision (W): 86.78%
Recall (W): 90.91%
F1 (W): 88.80%
Precision (Macro): 60.77%
Recall (Macro): 63.58%
F1 (Macro): 62.14%


# **Deep learning**


*   BPNN
*   CNN
*   Autoencoder Classifier





In [ ]:
# DEEP LEARNING MODELS

import numpy as np
import tensorflow as tf
import random
import os

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# FIXED SEED
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

os.environ['TF_DETERMINISTIC_OPS'] = '1'

# SETTINGS
EPOCHS = 40
BATCH_SIZE = 32

# CLASS WEIGHTS
def get_class_weights(y):
    classes = np.unique(y)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
    return dict(zip(classes, weights))


def build_bpnn(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

def build_cnn(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Reshape((input_dim, 1)),
        tf.keras.layers.Conv1D(16, 3, padding='same', activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

# AUTOENCODER + CLASSIFIER
def build_autoencoder_classifier(input_dim, num_classes):

    inputs = tf.keras.Input(shape=(input_dim,))

    encoded = tf.keras.layers.Dense(64, activation='relu')(inputs)
    encoded = tf.keras.layers.Dense(32, activation='relu')(encoded)

    decoded = tf.keras.layers.Dense(64, activation='relu')(encoded)
    decoded = tf.keras.layers.Dense(input_dim, activation='linear', name="reconstruction")(decoded)

    classifier = tf.keras.layers.Dense(num_classes, activation='softmax', name="classification")(encoded)

    model = tf.keras.Model(inputs=inputs, outputs=[decoded, classifier])

    return model

# TRAINING FUNCTIONS
def train_standard(model_fn, X_train, y_train, X_test, y_test):

    num_classes = len(np.unique(y_train))
    input_dim = X_train.shape[1]

    model = model_fn(input_dim, num_classes)

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    class_weights = get_class_weights(y_train)

    model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        class_weight=class_weights,
        shuffle=True
    )

    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    return evaluate(y_test, y_pred)

def train_autoencoder(X_train, y_train, X_test, y_test):

    num_classes = len(np.unique(y_train))
    input_dim = X_train.shape[1]

    model = build_autoencoder_classifier(input_dim, num_classes)

    model.compile(
        optimizer='adam',
        loss={
            "reconstruction": "mse",
            "classification": "sparse_categorical_crossentropy"
        },
        loss_weights={
            "reconstruction": 0.3,
            "classification": 1.0
        },
        metrics={"classification": "accuracy"}
    )

    model.fit(
        X_train,
        {"reconstruction": X_train, "classification": y_train},
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        shuffle=True
    )

    preds = model.predict(X_test, verbose=0)[1]
    y_pred = np.argmax(preds, axis=1)

    return evaluate(y_test, y_pred)

# MAIN LOOP
for fs_name, features in fs_methods.items():

    print(f"\n\n========== DEEP LEARNING | FEATURE SET: {fs_name} ==========")

    # SCALE
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train[features])
    X_test_s = scaler.transform(X_test[features])

    # BPNN
    print("\nBPNN")
    print_results(train_standard(build_bpnn, X_train_s, y_train, X_test_s, y_test))

    # CNN
    print("\nCNN")
    print_results(train_standard(build_cnn, X_train_s, y_train, X_test_s, y_test))

    # AUTOENCODER
    print("\nAutoencoder Classifier")
    print_results(train_autoencoder(X_train_s, y_train, X_test_s, y_test))



========== DEEP LEARNING | FEATURE SET: RFE_SVM ==========

FCN
Accuracy: 86.36%
Precision (W): 83.70%
Recall (W): 86.36%
F1 (W): 85.01%
Precision (Macro): 58.55%
Recall (Macro): 60.49%
F1 (Macro): 59.50%

BPNN
Accuracy: 89.39%
Precision (W): 88.85%
Recall (W): 89.39%
F1 (W): 88.99%
Precision (Macro): 76.93%
Recall (Macro): 72.94%
F1 (Macro): 74.32%

CNN
Accuracy: 89.39%
Precision (W): 88.85%
Recall (W): 89.39%
F1 (W): 88.99%
Precision (Macro): 76.93%
Recall (Macro): 72.94%
F1 (Macro): 74.32%

Autoencoder Classifier
Accuracy: 87.88%
Precision (W): 83.89%
Recall (W): 87.88%
F1 (W): 85.84%
Precision (Macro): 58.65%
Recall (Macro): 61.37%
F1 (Macro): 59.98%


========== DEEP LEARNING | FEATURE SET: Mutual_Info ==========

FCN
Accuracy: 86.36%
Precision (W): 83.70%
Recall (W): 86.36%
F1 (W): 85.01%
Precision (Macro): 58.55%
Recall (Macro): 60.49%
F1 (Macro): 59.50%

BPNN
Accuracy: 84.85%
Precision (W): 82.27%
Recall (W): 84.85%
F1 (W): 83.51%
Precision (Macro): 57.34%
Recall (Macro): 59.

# **Transformer-based Tabular models**

*   FT-Transformer
*   TabTransformer
*   TabNet



In [ ]:
# TRANSFORMER

import numpy as np

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, BaggingClassifier
from sklearn.linear_model import LogisticRegression

# CV (STANDARDIZED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# TRANSFORMER-STYLE MODELS
transformer_models = {

    # FT-Transformer
    "FT-Transformer": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPClassifier(max_iter=1500, early_stopping=True, random_state=42))
        ]),
        {
            "mlp__hidden_layer_sizes": [(256,128,64), (128,128)],
            "mlp__alpha": [0.0001, 0.001]
        }
    ),

    # TabTransformer
    "TabTransformer": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPClassifier(max_iter=1200, early_stopping=True, random_state=42))
        ]),
        {
            "mlp__hidden_layer_sizes": [(128,64), (256,128)],
        }
    ),
}

# ------------------------------
# LOOP
# ------------------------------
for fs_name, features in fs_methods.items():

    print(f"\n\n========== TRANSFORMER | FEATURE SET: {fs_name} ==========")

    X_train_sel = X_train[features]
    X_test_sel = X_test[features]

    # TRANSFORMER MODELS
    for name, (model, params) in transformer_models.items():

        print(f"\n{name}")

        total_space = int(np.prod([len(v) for v in params.values()]))
        n_iter = min(5, total_space)

        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=params,
            n_iter=n_iter,
            cv=cv,
            scoring='accuracy',
            n_jobs=-1,
            random_state=42
        )

        # TRAIN (NO LEAKAGE)
        search.fit(X_train_sel, y_train)

        # TEST
        y_pred = search.predict(X_test_sel)

        print("Best Params:", search.best_params_)
        print_results(evaluate(y_test, y_pred))




========== TRANSFORMER | FEATURE SET: RFE_SVM ==========

FT-Transformer (Approx)
Best Params: {'mlp__hidden_layer_sizes': (256, 128, 64), 'mlp__alpha': 0.0001}
Accuracy: 87.88%
Precision (W): 83.98%
Recall (W): 87.88%
F1 (W): 85.82%
Precision (Macro): 58.40%
Recall (Macro): 61.82%
F1 (Macro): 60.01%

TabTransformer (Approx)
Best Params: {'mlp__hidden_layer_sizes': (256, 128)}
Accuracy: 80.30%
Precision (W): 77.25%
Recall (W): 80.30%
F1 (W): 78.20%
Precision (Macro): 54.50%
Recall (Macro): 54.70%
F1 (Macro): 54.20%

Attention-like Boosting (HistGB)
Best Params: {'max_depth': 5, 'learning_rate': 0.05}
Accuracy: 89.39%
Precision (W): 86.64%
Recall (W): 89.39%
F1 (W): 87.99%
Precision (Macro): 60.68%
Recall (Macro): 62.70%
F1 (Macro): 61.68%

SUPER ENSEMBLE (RF + BBC + LR)
Accuracy: 87.88%
Precision (W): 90.16%
Recall (W): 87.88%
F1 (W): 88.84%
Precision (Macro): 68.86%
Recall (Macro): 72.06%
F1 (Macro): 69.84%


========== TRANSFORMER | FEATURE SET: Mutual_Info ==========

FT-Transform

In [5]:
# TABNET MODEL ONLY

import numpy as np
import pandas as pd
import random
import os
import warnings

from collections import Counter

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin

# ============================================
# SUPPRESS TABNET WARNING
# ============================================

warnings.filterwarnings(
    "ignore",
    message="No early stopping will be performed"
)

# ============================================
# INSTALL TABNET IF MISSING
# ============================================

import sys
import subprocess

def install(package):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", package]
    )

try:
    from pytorch_tabnet.tab_model import TabNetClassifier
except ModuleNotFoundError:
    install("pytorch-tabnet")
    from pytorch_tabnet.tab_model import TabNetClassifier

# ============================================
# FIXED SEED
# ============================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================
# SAFE CV
# ============================================

min_class_count = min(Counter(y_train).values())

cv_splits = max(2, min(5, min_class_count))

print(f"Using {cv_splits}-Fold Stratified CV")

cv = StratifiedKFold(
    n_splits=cv_splits,
    shuffle=True,
    random_state=SEED
)

# ============================================
# TABNET WRAPPER
# ============================================

class TabNetWrapper(BaseEstimator, ClassifierMixin):

    def __init__(
        self,
        n_d=16,
        n_a=16,
        n_steps=3,
        gamma=1.3,
        lambda_sparse=1e-4
    ):

        self.n_d = n_d
        self.n_a = n_a
        self.n_steps = n_steps
        self.gamma = gamma
        self.lambda_sparse = lambda_sparse

    def fit(self, X, y):

        # Convert dataframe to numpy
        X_np = X.values if hasattr(X, "values") else X

        # Internal validation split
        X_train_inner, X_valid_inner, y_train_inner, y_valid_inner = train_test_split(
            X_np,
            y,
            test_size=0.1,
            stratify=y,
            random_state=SEED
        )

        # Create model
        self.model_ = TabNetClassifier(
            n_d=self.n_d,
            n_a=self.n_a,
            n_steps=self.n_steps,
            gamma=self.gamma,
            lambda_sparse=self.lambda_sparse,
            seed=SEED,
            verbose=0
        )

        # Train with early stopping
        self.model_.fit(
            X_train=X_train_inner,
            y_train=y_train_inner,

            eval_set=[
                (X_valid_inner, y_valid_inner)
            ],

            eval_metric=["accuracy"],

            max_epochs=100,
            patience=20,

            batch_size=32,
            virtual_batch_size=16
        )

        return self

    def predict(self, X):

        X_np = X.values if hasattr(X, "values") else X

        return self.model_.predict(X_np)

    def predict_proba(self, X):

        X_np = X.values if hasattr(X, "values") else X

        return self.model_.predict_proba(X_np)

# ============================================
# PIPELINE
# ============================================

tabnet_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", TabNetWrapper())
])

# ============================================
# PARAMETER SPACE
# ============================================

param_dist = {

    "model__n_d": [8, 16, 32],

    "model__n_a": [8, 16, 32],

    "model__n_steps": [3, 5],

    "model__gamma": [1.0, 1.3],

    "model__lambda_sparse": [1e-3, 1e-4]
}

# ============================================
# MAIN LOOP
# ============================================

for fs_name, features in fs_methods.items():

    print(f"\n\n========== TABNET | FEATURE SET: {fs_name} ==========")

    # Feature selection
    X_train_sel = X_train[features]
    X_test_sel = X_test[features]

    # Prevent n_iter warning
    total_space = int(
        np.prod([len(v) for v in param_dist.values()])
    )

    n_iter = min(10, total_space)

    # Randomized Search
    search = RandomizedSearchCV(
        estimator=tabnet_pipeline,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=cv,
        scoring='accuracy',
        n_jobs=1,
        random_state=SEED
    )

    # TRAIN
    search.fit(X_train_sel, y_train)

    # TEST
    y_pred = search.predict(X_test_sel)

    # RESULTS
    print("Best Params:", search.best_params_)

    print_results(
        evaluate(y_test, y_pred)
    )

Using 5-Fold Stratified CV


========== TABNET | FEATURE SET: ANOVA ==========

Early stopping occurred at epoch 23 with best_epoch = 3 and best_val_0_accuracy = 0.95455


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 38 with best_epoch = 18 and best_val_0_accuracy = 0.95455


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 26 with best_epoch = 6 and best_val_0_accuracy = 0.95455


...

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 30 with best_epoch = 10 and best_val_0_accuracy = 0.90909


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 34 with best_epoch = 14 and best_val_0_accuracy = 0.92593
Best Params: {'model__n_steps': 3, 'model__n_d': 8, 'model__n_a': 32, 'model__lambda_sparse': 0.001, 'model__gamma': 1.0}
Accuracy: 90.91%
Precision (W): 86.78%
Recall (W): 90.91%
F1 (W): 88.80%
Precision (Macro): 60.77%
Recall (Macro): 63.58%
F1 (Macro): 62.14%


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
